# ADA Data Manipulation - Category System

## How to Add New Categories

### Step 1: Update your tariff-hscodes CSV
Add entries with your new category names:
```csv
HS Code,Category
440710,Lumber (old)
440910,Lumber (new)
...
```

### Step 2: Update CATEGORY_CONFIG (Cell 3)
Map each CSV category name to a short column prefix:
```python
CATEGORY_CONFIG = {
    'Auto': 'Auto',
    'Aluminum': 'Alum',
    'Lumber (old)': 'LumOld',   # <-- Add new categories
    'Lumber (new)': 'LumNew',   # <-- Add new categories
    ...
}
```

### Step 3: Run all cells
The notebook will automatically:
- Create NAICS sets for each category
- Generate `{Prefix}_B`, `{Prefix}_E` columns (weighted business/employee counts)
- Generate `{Prefix}_1`, `{Prefix}_2`, `{Prefix}_3` columns (percentages for choropleth)
- Export everything to GeoJSON, Shapefile, and CSV

### Output Columns per Category
- `{Prefix}_B` = Weighted number of businesses
- `{Prefix}_E` = Weighted number of employees (by work location)
- `{Prefix}_C` = Weighted number of employees (by residence)
- `{Prefix}_1` = % of all businesses affected
- `{Prefix}_2` = % of all employees affected (by work location)
- `{Prefix}_3` = % of census population in affected jobs (by residence)

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
from shapely.ops import unary_union
from tqdm import tqdm
import json
import gc
import time
from datetime import timedelta

## Configuration: Define your tariff categories here

In [ ]:
# ============================================================
# CATEGORY CONFIGURATION - Edit this to add/modify categories
# ============================================================

# Path to the tariff-hscodes CSV file
TARIFF_HSCODES_FILE = '../raw/tariffs/tariff_hs_codes_9_29_2026.csv'

# Category configuration: maps CSV category names to short column prefixes
# Format: 'Category Name in CSV': 'Short_Prefix_for_Columns'

CATEGORY_CONFIG = {
    'Auto': 'Auto',
    'Aluminum': 'Alum',
    'Steel': 'Steel',
    'Copper': 'Cop',
    'Energy Mineral': 'Ene',
    'MHDV': 'MHDV',
    'Lumber (old)': 'LumOld',
    'Lumber (new)': 'LumNew',
    'Dairy': 'Dairy',
    'Alcohol': 'Alcohol',
    'Motor': 'Motor',
    'before September 29': 'before September 29',
    'after September 29': 'after September 29',
    'Section 338 - Tariffed': 'S338_Tar',
    'Section 338 - Excluded': 'S338_Exc',
    'Section 338 - Total': 'S338_Tot',
}

# Always-included special categories (don't change these unless you know what you're doing)
SPECIAL_CATEGORIES = ['nonCUSMA', 'Total']  # nonCUSMA = goods not covered by CUSMA

print(f"✅ Configured {len(CATEGORY_CONFIG)} tariff categories: {list(CATEGORY_CONFIG.keys())}")
print(f"   Using tariff file: {TARIFF_HSCODES_FILE}")

# STEP 3: Connecting Tariffed HS Codes, NAICS Codes and respective CUSMA Non-Utilisation Rates via Concordance Table

In [ ]:
tariffed = pd.read_csv(TARIFF_HSCODES_FILE, encoding_errors='ignore', dtype={'HS Code': str})

tariffed['HS_Code_6digit'] = (
    tariffed['HS Code']
    .str.replace('.', '', regex=False)
    .str[:6]
)

tariffed = tariffed[['HS_Code_6digit', 'Category']].drop_duplicates()

# Validate that all categories in CSV are in our config
csv_categories = set(tariffed['Category'].dropna().unique())
configured_categories = set(CATEGORY_CONFIG.keys())
unknown_categories = csv_categories - configured_categories

if unknown_categories:
    print(f"⚠️ WARNING: Found categories in CSV not in CATEGORY_CONFIG: {unknown_categories}")
    print("   Add them to CATEGORY_CONFIG or they will be treated as 'nonCUSMA'")
else:
    print(f"✅ All CSV categories are configured: {csv_categories}")

In [ ]:
concordance = pd.read_csv('../raw/C616_HS8toNaics6_concord_202505.csv', dtype={'hts10': str})

concordance['HS_Code_6digit'] = (
    concordance['HS8 Code']
    .astype(str)
    .str.zfill(8)
    .str[:6]
)

concordance['HS_Code_2digit'] = (
    concordance['HS8 Code']
    .astype(str)
    .str.zfill(8)
    .str[:2]
)

concordance['NAICS'] = concordance['NAICS 6 Code'].astype(str)

concordance = concordance[['HS_Code_6digit', 'NAICS', 'HS_Code_2digit']].drop_duplicates()

print(concordance[concordance['HS_Code_6digit'] == '440311'].head())

In [ ]:
util = pd.read_csv('../raw/USMCA Utilization Data.csv')

util['HS_Code_2digit'] = (
    util['HS Classification']
    .astype(str)
    .str[:2]
)

util['nonutil_rate'] = util['USMCA_Nonutilisation_May2025']

util = util[['HS_Code_2digit', 'nonutil_rate']]

Setting the non-utilisation rate for those with sectoral tariffs at 1 reflects the fact that the 35% tariffs on non-CUSMA goods does not apply to the sectoral tariffs and that producers impacted by sectoral tariffs cannot use CUSMA to get their goods tariff-free.

In [ ]:
naics_tar = concordance.merge(tariffed, on='HS_Code_6digit', how='left')

counts = naics_tar['HS_Code_6digit'].value_counts().reset_index()
naics_tarc = naics_tar.merge(counts, on='HS_Code_6digit', how='left')

naics_imp = naics_tarc.merge(util, on='HS_Code_2digit', how='left')

# new column non-util chapter getting the nonutil rate! (for before section 338)
naics_imp['nonutil_chapter'] = naics_imp['nonutil_rate'] 

# Making the codes with a sectoral tariff category assigned to have a non-util rate value of 1
# This ensures that when multiplied later, sectoral tariffs do not affect the CUSMA non-utilisation rates to compute the impact of nonCUSMA 35% tariffs
naics_imp.loc[naics_imp['Category'].notna(), 'nonutil_rate'] = 1
naics_imp['Category'] = naics_imp['Category'].fillna('nonCUSMA')
naics_imp

# STEP 4: Getting Weights by Province/Territory

Since multiple NAICS codes may contribute to the production of one HS code product, and we do not how much of a part does each NAICS contribute to the whole HS code good production, **thus an assumption is made to divide them equally**. Hence, when each export value is added, it is divided them by the count (how many times does that HS Code get repeated).  

Meanwhile, the non-utilisation rate of CUSMA exemption by each HS Code is first multiplied to the total value of each HS Code export to the US, before divided by the count.

In [ ]:
EXPORT_DIR = '../raw/exports'

# Initialize the DataFrame with NAICS data
tariff_exp_val = naics_imp.copy()

# Define all regions to process
provinces = ['NL', 'PEI', 'NS', 'NB', 'QC', 'ON', 'MB', 'SK', 'AL', 'BC', 'YK', 'NWT', 'NU']
cols = ['Commodity', 'Value ($)']

for province in provinces:
    # Process Global data
    global_df = pd.read_csv(f'{EXPORT_DIR}/{province}-Global.csv', usecols=cols)
    global_df['HS_Code_6digit'] = global_df['Commodity'].str.replace('.', '', regex=False).str[:6]
    global_df[f'{province}_Global'] = global_df['Value ($)']
    global_df = global_df[['HS_Code_6digit', f'{province}_Global']]
    
    tariff_exp_val = tariff_exp_val.merge(global_df, on='HS_Code_6digit', how='left')
    tariff_exp_val[f'{province}_Global'] = tariff_exp_val[f'{province}_Global'] / tariff_exp_val['count']
    
    # Process US data
    us_df = pd.read_csv(f'{EXPORT_DIR}/{province}-US.csv', usecols=cols)
    us_df['HS_Code_6digit'] = us_df['Commodity'].str.replace('.', '', regex=False).str[:6]
    us_df[f'{province}_US'] = us_df['Value ($)']
    us_df = us_df[['HS_Code_6digit', f'{province}_US']]
    
    tariff_exp_val = tariff_exp_val.merge(us_df, on='HS_Code_6digit', how='left')
    tariff_exp_val[f'{province}_US_raw'] = tariff_exp_val[f'{province}_US'] / tariff_exp_val['count']
    ## NOW HAS THE NON UTIL RATE AS WELL!
    tariff_exp_val[f'{province}_US'] = tariff_exp_val[f'{province}_US_raw'] * tariff_exp_val['nonutil_rate']

# Drop the non-relevant columns
tariff_exp_val = tariff_exp_val.drop(columns=['HS_Code_2digit', 'count', 'nonutil_rate'])

tariff_exp_val = tariff_exp_val.fillna(0)

tariff_exp_val

In [ ]:
# Step 1: Aggregate ONLY the necessary sums (Global/US columns)
aggregates = {
    **{f'{province}_Global': (f'{province}_Global', 'sum') for province in provinces},
    **{f'{province}_US': (f'{province}_US', 'sum') for province in provinces},
}

weight_naics = tariff_exp_val.groupby('NAICS', as_index=False).agg(**aggregates)

# Step 2: Compute rates and keep ONLY those columns
for province in provinces:
    global_col = f'{province}_Global'
    us_col = f'{province}_US'
    rate_col = f'{province}'
    
    weight_naics[rate_col] = (
        weight_naics[us_col] / weight_naics[global_col]
    )

# Step 3: Select only naics, naics_2digit, and rate columns
final_columns = ['NAICS'] + [f'{province}' for province in provinces]
weight_naics = weight_naics[final_columns]
weight_naics = weight_naics.fillna(0)

# Result
weight_naics

In [ ]:
# Categories in force BEFORE Aug 22, 2026 (exact strings from CATEGORY_CONFIG)
IN_FORCE_BEFORE = [
    'Auto', 'Aluminum', 'Steel', 'Copper', 'Energy Mineral', 'MHDV',
    'Lumber (old)', 'Lumber (new)', 'before September 29',
]

# An HS code keeps rate 1.0 if ANY of its tags was in force before Aug 22.
# Otherwise the whole code falls back to its chapter non-utilisation rate.
hs_in_force = set(
    tariff_exp_val.loc[tariff_exp_val['Category'].isin(IN_FORCE_BEFORE), 'HS_Code_6digit']
)

t = tariff_exp_val.copy()
rate_sb = np.where(t['HS_Code_6digit'].isin(hs_in_force), 1.0, t['nonutil_chapter'])

for province in provinces:
    t[f'{province}_US'] = t[f'{province}_US_raw'] * rate_sb

weight_naics_sb = t.groupby('NAICS', as_index=False).agg(**aggregates)
for province in provinces:
    weight_naics_sb[province] = weight_naics_sb[f'{province}_US'] / weight_naics_sb[f'{province}_Global']
weight_naics_sb = weight_naics_sb[['NAICS'] + provinces].fillna(0)

weight_naics_sb

In [ ]:
hs_ba = tariff_exp_val.loc[tariff_exp_val['Category'] == 'before August 22', 'HS_Code_6digit']
print(tariff_exp_val[tariff_exp_val['HS_Code_6digit'].isin(hs_ba)]['Category'].value_counts())

In [ ]:
sb = weight_naics_sb.set_index('NAICS')['ON']
base = weight_naics.set_index('NAICS')['ON']
d = (sb - base).dropna()
print(f"above base: {(d > 1e-12).sum()}   (must be 0)")
print(f"below base: {(d < -1e-12).sum()}")

# STEP 5: Using filtered NAICS Codes to filter directly-impacted businesses and estimate directly-impacted employees

In [ ]:
# DYNAMICALLY CREATE NAICS SETS FOR EACH CATEGORY

# Dictionary to store NAICS codes for each category
category_naics = {}

# Create NAICS sets for each configured category
for csv_name, short_name in CATEGORY_CONFIG.items():
    category_naics[short_name] = set(
        naics_imp[naics_imp['Category'] == csv_name]['NAICS'].unique()
    )
    print(f"  {short_name}: {len(category_naics[short_name])} NAICS codes")

# Special categories (always included)
category_naics['CUSMA'] = set(naics_imp[naics_imp['Category'] == 'nonCUSMA']['NAICS'].unique())
category_naics['Total'] = set(naics_imp['NAICS'].unique())

print(f"\n✅ Created {len(category_naics)} NAICS sets")
print(f"   Total unique NAICS codes: {len(category_naics['Total'])}")

# For backward compatibility, also create individual variables (optional)
total_naics = category_naics['Total']

In [ ]:
SCENARIOS = {
    'ScenBefore': ['before September 29', 'nonCUSMA'],
    'ScenAfter':  ['after September 29',  'nonCUSMA'],
}
# if step 1 returned 0, extend each list with the sectoral categories in force

for name, cats in SCENARIOS.items():
    category_naics[name] = set().union(*(
        set(naics_imp.loc[naics_imp['Category'] == c, 'NAICS'].unique()) for c in cats
    ))

In [ ]:
print(category_naics['ScenAfter'] == category_naics['Total'])

In [ ]:
# SB, SA = category_naics['ScenBefore'], category_naics['ScenAfter']
# S338, nonC = category_naics['Section 338'], category_naics['CUSMA']

# print(f"ScenAfter adds {len(SA - SB)} NAICS over ScenBefore")
# print(f"S338 NAICS not already non-CUSMA: {len(S338 - nonC)} of {len(S338)}")
# print(f"ScenAfter == Total: {SA == category_naics['Total']}")

In [ ]:
added = category_naics['ScenAfter'] - category_naics['ScenBefore']
print(sorted(added))
print(naics_imp[naics_imp['NAICS'].isin(added)][['NAICS','Category']].drop_duplicates())

In [ ]:
added = category_naics['ScenAfter'] - category_naics['ScenBefore']
rows = naics_imp[naics_imp['NAICS'].isin(added)]
print(rows[['NAICS','HS_Code_6digit','Category']].drop_duplicates().sort_values('NAICS'))

print(naics_tarc['Category'].isna().sum(), "untagged rows")
print(naics_tarc['Category'].notna().sum(), "tagged rows")

In [ ]:
lum = category_naics['LumOld'] | category_naics['LumNew']
print(f"lumber NAICS not already non-CUSMA: {len(lum - category_naics['CUSMA'])} of {len(lum)}")

In [ ]:
# ============================================================
# DYNAMICALLY CREATE COLUMN STRUCTURE
# ============================================================

# Base columns (always present)
col_i = ['DA', 'All_Businesses', 'All_Employees']

# Add columns for each category (Business and Employee counts)
all_category_prefixes = list(CATEGORY_CONFIG.values()) + ['CUSMA', 'Total'] + list(SCENARIOS.keys())
for prefix in all_category_prefixes:
    col_i.extend([f'{prefix}_B', f'{prefix}_E'])

# Add individual NAICS codes as column headers for Est_Employees by NAICS
col_i.extend(sorted(total_naics))

business = pd.DataFrame(columns = col_i)

print(f"✅ Created DataFrame with {len(col_i)} columns")
print(f"   Category columns: {[p for p in all_category_prefixes]}")

In [ ]:
# ============================================================
# MAIN PROCESSING LOOP - DYNAMIC CATEGORIES
# ============================================================

chunk_size = 1_000_000
province_code = {10:'NL',11:'PEI',12:'NS',13:'NB',24:'QC',35:'ON',46:'MB',47:'SK',48:'AL',59:'BC',60:'YK',61:'NWT',62:'NU'}

# melting weights in long form to prepare for a vectorized merge
wlong = (
    weight_naics
    .melt(id_vars='NAICS', var_name='Province', value_name='Rate')
)

wlong = wlong.merge(
    weight_naics_sb.melt(id_vars='NAICS', var_name='Province', value_name='Rate_SB'),
    on=['NAICS', 'Province'], how='outer'
)

# loading necessary data
all_cols = pd.read_csv('../input-data/large_size_data/Dec2022_Estabcounts_byDA.csv', encoding='ISO-8859-1', nrows=1).columns
cols_to_keep = [c for c in all_cols if c != 'Without employees']

# suggesting dtypes for performance purposes
dtype_hint = {
    '1-4':'Int64','5-9':'Int64','10-19':'Int64','20-49':'Int64',
    '50-99':'Int64','100-199':'Int64','200-499':'Int64','500 +':'Int64',
    'Total, with employees':'Int64',
}

total_start = time.time()
chunk_num = 0

# Preparing two empty lists
agg_frames = []       # List for weighted amount of businesses and est employees for each tariff
per_naics_frames = [] # List for total amount of employees for each NAICS code in each ADA --> needed for Step 8 later

# Build the list of aggregation columns dynamically
agg_cols = ['All_Businesses', 'All_Employees']
for prefix in all_category_prefixes:
    agg_cols.extend([f'{prefix}_B', f'{prefix}_E'])

for chunk in pd.read_csv(
        '../input-data/large_size_data/Dec2022_Estabcounts_byDA.csv',
        encoding='ISO-8859-1',
        chunksize=chunk_size,
        usecols=cols_to_keep,
        dtype=dtype_hint,
    ):
    t0 = time.time()
    chunk_num += 1

    # 1) Filter non-relevant rows
    chunk = chunk[~chunk['NAICS'].isin(['Sub-total, classified', 'Unclassified', 'Total'])].copy()

    # 2) Basic transforms (vectorized)
    # keep NAICS 6-digit as string
    chunk['NAICS'] = chunk['NAICS'].astype(str).str[:6]
    chunk['Business_per_NAICS'] = chunk['Total, with employees'].fillna(0)

    # estimate employees (vectorized)
    chunk['Est_Employees'] = (
        chunk['1-4'].fillna(0) * 3  +
        chunk['5-9'].fillna(0) * 7  +
        chunk['10-19'].fillna(0) * 15 +
        chunk['20-49'].fillna(0) * 35 +
        chunk['50-99'].fillna(0) * 75 +
        chunk['100-199'].fillna(0) * 150 +
        chunk['200-499'].fillna(0) * 350 +
        chunk['500 +'].fillna(0) * 550
    )

    # Province lookup
    # if 'DisseminationAre' isn't numeric, ensure this still works (it uses first two chars)
    chunk['ProvinceCode'] = chunk['DisseminationAre'].astype(str).str[:2].astype(int, errors='ignore')
    chunk['Province'] = pd.Series(chunk['ProvinceCode']).map(province_code)

    # 3) Merge the per-(NAICS, Province) Rate (vectorized, no apply)
    merged = chunk.merge(wlong, how='left', on=['NAICS','Province'])

    # 4) Weighted columns (vectorized)
    merged['Weighted_Business']  = np.ceil(merged['Business_per_NAICS'] * merged['Rate'])
    merged['Weighted_Employees'] = np.ceil(merged['Est_Employees'] * merged['Rate'])

    # 5) DYNAMIC CATEGORY MASKS - Create masks for each category
    s = merged['NAICS']
    wb = merged['Weighted_Business']
    we = merged['Weighted_Employees']

    # 6) the rate with non section 338 weights (original)
    wb_sb = np.ceil(merged['Business_per_NAICS'] * merged['Rate_SB'])
    we_sb = np.ceil(merged['Est_Employees'] * merged['Rate_SB'])
    
    # Apply masks dynamically for each category
    for prefix, naics_set in category_naics.items():
        is_in_category = s.isin(naics_set) if len(naics_set) else pd.Series(False, index=s.index)
        # new
        b, e = (wb_sb, we_sb) if prefix == 'ScenBefore' else (wb, we)
        merged[f'{prefix}_B'] = np.where(is_in_category, b, 0)
        merged[f'{prefix}_E'] = np.where(is_in_category, e, 0)

    # Always aggregate the unweighted totals too
    merged['All_Businesses'] = merged['Business_per_NAICS']
    merged['All_Employees']  = merged['Est_Employees']

    # 6) Chunk-level aggregation in ONE groupby
    by_da = merged.groupby('DisseminationAre', as_index=False)[agg_cols].sum()

    # 7) Generating the data for second list --> number of jobs per NAICS in each DA
    is_total = s.isin(total_naics)
    per_naics = (
        merged.loc[is_total, ['DisseminationAre','NAICS','Est_Employees']]
        .pivot_table(index='DisseminationAre', columns='NAICS', values='Est_Employees',
                     aggfunc='sum', fill_value=0)
        .reset_index()
    )

    # Saving the data into the two different lists in each chunk
    agg_frames.append(by_da)
    per_naics_frames.append(per_naics)

    print(f"Chunk {chunk_num} processed in {time.time()-t0:.2f} sec")

# Combine all chunks together to form one big dataframe
agg_all = pd.concat(agg_frames, ignore_index=True).groupby('DisseminationAre', as_index=False).sum()

per_naics_all = pd.concat(per_naics_frames, ignore_index=True).groupby('DisseminationAre', as_index=False).sum()

business = agg_all.merge(per_naics_all, on='DisseminationAre', how='left')

business = business.rename(columns={'DisseminationAre':'DA'})

print(f"\n✅ All chunks processed in {time.time()-total_start:.2f} sec")
print(f"   Processed categories: {list(category_naics.keys())}")

business

# STEP 6: Regrouping filtered data into ADAs

In [ ]:
da = gpd.read_file('../input-data/large_size_data/lda_000b21a_e.shp')
da ['DA'] = da ['DAUID']
da ['DADGUID'] = da ['DGUID']
da = da[['DA', 'DADGUID']]

ada = gpd.read_file('../input-data/large_size_data/lada000b21a_e.shp')
adac = ada.copy()
adac ['ADADGUID'] = adac ['DGUID']
adac = adac[['ADADGUID', 'geometry']]

relation = pd.read_csv('../intermediate/ada_da_relation.csv')

In [ ]:
da_relation = da.merge(relation, on='DADGUID', how='left')
full_relation = da_relation.merge(adac, on='ADADGUID', how='left')
full_relation['DA'] = pd.to_numeric(full_relation['DA'], errors='coerce').astype('Int64')

In [ ]:
# ============================================================
# AGGREGATE TO ADA LEVEL - DYNAMIC CATEGORIES
# ============================================================

business_merged =(
    business.merge(full_relation, on='DA', how='right')
) 

# Separate numeric columns from geometry
numeric_cols = [col for col in business.columns if col != 'DA']

# Fill NA only for numeric columns, then cast to int64
for col in numeric_cols:
    business_merged[col] = business_merged[col].fillna(0).astype('int64')

# Build aggregation dictionary dynamically
agg_dict = {
    'All_Businesses': ('All_Businesses', 'sum'),
    'All_Employees': ('All_Employees', 'sum'),
    'geometry': ('geometry', 'first')
}

# Add category columns dynamically
for prefix in all_category_prefixes:
    agg_dict[f'{prefix}_B'] = (f'{prefix}_B', 'sum')
    agg_dict[f'{prefix}_E'] = (f'{prefix}_E', 'sum')

# Add dynamic aggregation rules for each NAICS code
for naics in total_naics:
    agg_dict[naics] = (naics, 'sum')

# Perform grouped aggregation
business_grouped = business_merged.groupby('ADADGUID', as_index=False).agg(**agg_dict)

print(f"✅ Aggregated to {len(business_grouped)} ADAs with {len(all_category_prefixes)} category pairs")

business_grouped

# STEP 7: Processing it into Centroids for Counts and Choropleth for Rates

Since the earlier table shows the *weighted* numbers of directly exposed businesses and employees (by work location) together with *total* number of employees (by work location) for each affected NAICS code, the former is separated from the latter. The former needs to be processed into separate GDFs for centroids (to show counts) and choropleths (to show percentages)

In [ ]:
# ============================================================
# SEPARATE WEIGHTED COLUMNS FROM NAICS COLUMNS - DYNAMIC
# ============================================================

# Build excluded columns list dynamically
excluded_cols = ['All_Businesses', 'All_Employees']
for prefix in all_category_prefixes:
    excluded_cols.extend([f'{prefix}_B', f'{prefix}_E'])
excluded_cols.append('geometry')

business_filter = business_grouped[['ADADGUID'] + [col for col in business_grouped.columns if col in excluded_cols]]

business_census = business_grouped[[col for col in business_grouped.columns if col not in excluded_cols]]

In [ ]:
# Convert to GeoDataFrame
cent_gdf = gpd.GeoDataFrame(business_filter, geometry='geometry', crs = 'EPSG:3347')

# Set a point within each polygon
cent_gdf = cent_gdf.drop(columns=['All_Businesses', 'All_Employees'])
cent_gdf['geometry'] = cent_gdf.geometry.representative_point()
cent_gdf.set_geometry('geometry', inplace=True)
cent_gdf

In [ ]:
# ============================================================
# CHOROPLETH PERCENTAGES - DYNAMIC CATEGORIES
# ============================================================

choro_cols = business_filter.copy()

# Use all category prefixes dynamically
tars = all_category_prefixes

for tar in tars:
    tar_1 = f'{tar}_1'
    tar_2 = f'{tar}_2'

    choro_cols[tar_1] = (
        choro_cols[f'{tar}_B']/choro_cols['All_Businesses']
    )

    choro_cols[tar_2] = (
        choro_cols[f'{tar}_E']/choro_cols['All_Employees']
    )

choro_cols = choro_cols[['ADADGUID'] + [f'{tar}_1' for tar in tars] + [f'{tar}_2' for tar in tars] + ['geometry']]

choro_gdf = gpd.GeoDataFrame(choro_cols, geometry = 'geometry', crs = 'EPSG:3347')
choro_gdf

# STEP 8: Generating Weight of Directly Exposed Jobs to Total Accessible Jobs in Each Industry

It is likely that while employees live close to their workplace, they do not live in the same ADA as they work in.  
  
StatsCan Census 2021 data shows a huge drop in the number of Canadians who travel more than 15km to their work vis-a-vis those who travel less than that distance to work.  
  
Thus, this cell creates a dictionary where for each ADA, it lists down, including itself, the ADA IDs within a 15km buffer around it (for small ADAs) or ADA IDs that are adjacent to it (for large ADAs). Small ADAs are defined as ADAs with an area less than (15km)^2.

In [ ]:
# # Copy from earlier ADA shapefile and ensure it's in projected CRS (EPSG:3347)
# adas = ada.to_crs("EPSG:3347")

# # Create a column to mark if ADA is "small" (<= 706 km2)
# adas['is_small'] = adas['LANDAREA'] <= 706

# # Build spatial index once
# adas_sindex = adas.sindex

# # Simplify geometries early to reduce memory use
# adas['geometry'] = adas['geometry'].simplify(100)

# # Prepare empty dictionary
# ada_neighbors = {}

# # Process in chunks to avoid RAM overload
# chunk_size = 500
# n = len(adas)

# # Track overall time
# start_time = time.time()

# for start in tqdm(range(0, n, chunk_size)):
#     chunk_start_time = time.time()
#     end = min(start + chunk_size, n)
#     chunk = adas.iloc[start:end].copy()

#     for idx, row in chunk.iterrows():
#         ada_uid = row['DGUID']
#         geom = row['geometry']
#         is_small = row['is_small']

#         if is_small:
#             buffer_geom = geom.buffer(15000)
#             possible_matches_index = list(adas_sindex.intersection(buffer_geom.bounds))
#             possible_matches = adas.iloc[possible_matches_index]
#             matches = possible_matches[possible_matches.geometry.intersects(buffer_geom)]
#         else:
#             possible_matches_index = list(adas_sindex.intersection(geom.bounds))
#             possible_matches = adas.iloc[possible_matches_index]
#             matches = possible_matches[possible_matches.geometry.touches(geom) | (possible_matches['DGUID'] == ada_uid)]

#         ada_neighbors[ada_uid] = matches['DGUID'].tolist()

#     del chunk
#     gc.collect()

#     # Print chunk timing
#     chunk_elapsed = time.time() - chunk_start_time
#     print(f"Chunk {start}-{end} processed in {timedelta(seconds=chunk_elapsed)}")

# # Overall timing
# total_elapsed = time.time() - start_time
# print(f"\nTotal time taken: {timedelta(seconds=total_elapsed)}")

# # Optionally save the dictionary to disk
# with open("ada_neighbors.json", "w") as f:
#     json.dump(ada_neighbors, f)

In [ ]:
with open('ada_neighbors.json') as f:
    ada_neighbors = json.load(f)

The dictionary is then used in conjunction with the *total* count of employees (by work location), as separated in Cell 13 above, to find out the likely number of jobs of each 6-digit NAICS, and total number of jobs, that are 'accessible' from each ADA --> going by the assumption of travel distance made by Canadians to go to work from Census 2021 data

In [ ]:
# Ensure 'ADADGUID' is the index for fast lookup
business_census_indexed = business_census.set_index('ADADGUID')

# Prepare list to collect results
aggregated_results = []

# Loop through ADA + its neighbors
for ada_id, neighbor_list in tqdm(ada_neighbors.items()):
    # Filter business_census rows for all neighbors
    rows = business_census_indexed.loc[business_census_indexed.index.intersection(neighbor_list)]
    
    # Sum across all rows (by column)
    summed = rows.sum()
    
    # Store result with ADA ID
    result = summed.to_dict()
    result['ADADGUID'] = ada_id
    
    aggregated_results.append(result)

# Convert to DataFrame
jobs = pd.DataFrame(aggregated_results)

This is to find out the rate of directly exposed jobs (6-digit NAICS) to total jobs in each industry (1-digit NAICS jobs) that are accessible from each ADA

In [ ]:
jobs_rate = jobs.copy()

cola = [col for col in jobs.columns if col != 'ADADGUID']

# Calculate summed groups by first digit of column name
jobs_rate['Sum1'] = jobs_rate[[col for col in cola if col.startswith('1')]].sum(axis=1)
jobs_rate['Sum2'] = jobs_rate[[col for col in cola if col.startswith('2')]].sum(axis=1)
jobs_rate['Sum3'] = jobs_rate[[col for col in cola if col.startswith('3')]].sum(axis=1)

# Compute share per column
for col in cola:
    col_rate = f'{col}_R'
    if col.startswith('1'):
        jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum1']
    elif col.startswith('2'):
        jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum2']
    elif col.startswith('3'):
        jobs_rate[col_rate] = jobs_rate[col] / jobs_rate['Sum3']
    else:
        jobs_rate[col_rate] = 0  # fallback in case of unexpected prefix

# Final filtered DataFrame: only ADADGUID and the *_R columns
jobs_rate = jobs_rate[['ADADGUID'] + [f'{col}_R' for col in cola]]

jobs_rate = jobs_rate.fillna(0)

jobs_rate

# STEP 9: Applying Jobs Weight to Census Data

Census 2021 Data reports residents' occupational NAICS code at the two-digit level

In [ ]:
census = pd.read_csv('../input-data/large_size_data/98-401-X2021012_English_CSV_data.csv', encoding='latin1')

census = census[census['CHARACTERISTIC_ID'].isin([2259, 2262, 2263, 2266])] # Only taking the relevant 2-digit NAICS codes from Census 2021 data

census['ADADGUID'] = census['DGUID']

census['ProvinceCode'] = census['ADADGUID'].str[9:11].astype(int)

census['Province'] = census['ProvinceCode'].map(province_code)

census['CHARACTERISTIC_NAME'] = (
    census['CHARACTERISTIC_NAME']
    .str.replace(' ', '', regex=False)
    .str[:2]
)

census = census[['ADADGUID', 'Province', 'CHARACTERISTIC_NAME', 'C1_COUNT_TOTAL']]

census_pivot = census.pivot_table(
    index=['ADADGUID', 'Province'],
    columns='CHARACTERISTIC_NAME',
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

census_pivot

Thus the weights from Step 8 is used to estimate how many employees (by primary residence) are working in industries directly exposed to tariffs

In [ ]:
# Merge on ADADGUID to align both datasets
merged = census_pivot.merge(jobs_rate, on='ADADGUID', how='right')  # or 'left' if census is base

# Start building the adjusted DataFrame
adjusted_jobs = merged[['ADADGUID', 'Province', 'To']].copy()

# Compute adjusted values
for col in cola:
    rate_col = f'{col}_R'
    prefix = col[:2]

    if prefix in ['21', '22']:
        source_col = '21'
    elif prefix in ['31', '32', '33']:
        source_col = '31'
    else:
        source_col = prefix

    adjusted_jobs[col] = np.ceil(merged[rate_col] * merged[source_col])

adjusted_jobs['Sum'] = adjusted_jobs.drop(columns=['ADADGUID', 'Province', 'To']).sum(axis=1)

adjusted_jobs = adjusted_jobs.fillna(0)

adjusted_jobs['Province'] = adjusted_jobs['Province'].mask(
    adjusted_jobs['Province'].isin([0, np.nan]),
    adjusted_jobs['ADADGUID'].str[9:11].astype(int).map(province_code)
)

adjusted_jobs

# STEP 10: Applying Export Weights to the Census Data

Export weights from Step 4 are applied to account for regional differences

In [ ]:
lw_sb = weight_naics_sb.melt(id_vars='NAICS', var_name='Province', value_name='Rate')
lw_sb['NAICS'] = lw_sb['NAICS'].astype(str) + '_r'
weight_naics_pivot_sb = lw_sb.pivot(index=['Province'], columns='NAICS', values='Rate').reset_index()

In [ ]:
# Step 1: Melt to long format
long_weight = weight_naics.melt(id_vars='NAICS', var_name='Province', value_name='Rate')

long_weight['NAICS'] = long_weight['NAICS'].astype(str) + '_r'

# Step 2: Pivot to wide format
weight_naics_pivot = long_weight.pivot(index=['Province'], columns='NAICS', values='Rate').reset_index()

weight_naics_pivot

In [ ]:
# census by jobs for sb

census_byjobs_sb = adjusted_jobs.merge(weight_naics_pivot_sb, on='Province', how='left')
for col in cola:
    census_byjobs_sb[col] = np.ceil(census_byjobs_sb[col] * census_byjobs_sb[f'{col}_r'])

In [ ]:
# ============================================================
# APPLY EXPORT WEIGHTS - DYNAMIC CATEGORIES
# ============================================================

# Step 1: Merge adjusted_jobs with weight_naics_pivot on Province
census_byjobs = adjusted_jobs.merge(weight_naics_pivot, on='Province', how='left')

# Step 2: Multiply each column by its corresponding rate
for col in cola:
    rate_col = f'{col}_r'
    
    census_byjobs[col] = np.ceil(census_byjobs[col] * census_byjobs[rate_col])

# Step 3: Keep only ADADGUID and updated values
census_byjobs = census_byjobs[['ADADGUID', 'To'] + cola]

# Define output dictionary dynamically
grouped_data = {
    'ADADGUID': census_byjobs['ADADGUID'],  # retain ADA ID
    'Census': census_byjobs['To'],
}

# Add category sums dynamically
for prefix, naics_set in category_naics.items():
    src = census_byjobs_sb if prefix == 'ScenBefore' else census_byjobs
    grouped_data[f'{prefix}_C'] = src[[col for col in cola if col in naics_set]].sum(axis=1)

# Create final grouped DataFrame
census_bytariffs = pd.DataFrame(grouped_data)
census_bytariffs

# STEP 11: Processing and adding the data to Choropleth and Centroid GDFs

Add the data on employees (by primary residence) to the GDFs produced in Step 7

In [ ]:
# ============================================================
# CENTROIDS EXPORT - DYNAMIC CATEGORIES
# ============================================================

centroids = cent_gdf.merge(census_bytariffs, on='ADADGUID', how='left')

# Build column list dynamically
centroid_cols = ['ADADGUID']
for tar in tars:
    centroid_cols.extend([f'{tar}_B', f'{tar}_E', f'{tar}_C'])
centroid_cols.append('geometry')

centroids = centroids[centroid_cols]
centroids = centroids.to_crs('EPSG:4326')
centroids.to_file('../outputs/geojson/centroids.geojson', driver='GeoJSON')
centroids.to_csv("../outputs/csv/counts_ada.csv", index=False)
centroids

In [ ]:
# ============================================================
# PERCENTAGE BY TARIFFS - DYNAMIC CATEGORIES
# ============================================================

perc_bytariffs = census_bytariffs.copy()

for tar in tars:
    tar_3 = f'{tar}_3'

    perc_bytariffs[tar_3] = (
        perc_bytariffs[f'{tar}_C']/perc_bytariffs['Census']
    )

perc_bytariffs = perc_bytariffs[['ADADGUID'] + [f'{tar}_3' for tar in tars]]
# # Clip values to max 1 (100%) for CUSMA and Total
# perc_bytariffs['CUSMA_3'] = perc_bytariffs['CUSMA_3'].clip(upper=1)
# perc_bytariffs['Total_3'] = perc_bytariffs['Total_3'].clip(upper=1)

for p in ['CUSMA', 'Total', 'ScenBefore', 'ScenAfter']:
    perc_bytariffs[f'{p}_3'] = perc_bytariffs[f'{p}_3'].clip(upper=1)
perc_bytariffs

In [ ]:
# ============================================================
# CHOROPLETH EXPORT - DYNAMIC CATEGORIES
# ============================================================

choropleth = choro_gdf.merge(perc_bytariffs, on='ADADGUID', how='right')

# Build column list dynamically
choro_export_cols = ['ADADGUID']
for tar in tars:
    choro_export_cols.extend([f'{tar}_1', f'{tar}_2', f'{tar}_3'])
choro_export_cols.append('geometry')

choropleth = choropleth[choro_export_cols]
choropleth = choropleth.to_crs('EPSG:4326')
choropleth

In [ ]:
choropleth.to_file('../outputs/geojson/choropleth.geojson', driver='GeoJSON')
choropleth.to_file('../outputs/geojson/choropleth.shp', driver='ESRI Shapefile')

In [ ]:
choropleth.drop(columns="geometry").to_csv("../outputs/csv/percents_ada.csv", index=False)

# CSV Trails

for double-checking purposes

In [ ]:
business_grouped.drop(columns='geometry').to_csv('../outputs/csv/trail.csv', index=False)
business_filter.drop(columns='geometry').to_csv('../outputs/csv/trail2.csv', index=False)
business_census.to_csv('trail3.csv', index=False)

In [ ]:
choro_cols.drop(columns='geometry').to_csv('../outputs/csv/trail3.csv', index=False)

In [ ]:
jobs.to_csv('../outputs/csv/trail5.csv')
jobs_rate.to_csv('../outputs/csv/trail6.csv')
adjusted_jobs.to_csv('../outputs/csv/trail7.csv')